# DeepAgents · Plan 模式（多 SubAgent 协作）

本 Notebook 演示如何用 **deepagents** 实现一套「先规划、再派发」的 Plan 模式：

1. 一个 **Planner 主管** agent 负责理解目标、拆分子任务、制定执行计划；
2. 主管通过内置的 `task` 工具把每个子任务派发给专用 **subagent**（研究员 / 写作者 / 编码者 / 通用执行者）；
3. subagent 在隔离上下文里独立执行，只回传最终结论；
4. 主管汇总所有结果，产出统一答复。

实现代码见同目录 `plan_agent.py`。

In [ ]:
import sys, os

# 把仓库根目录加入 path，以便 `from utils.std_model import base_model` 可用
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../../.."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("REPO_ROOT =", REPO_ROOT)

In [ ]:
from utils.std_model import base_model
from plan_agent import (
    build_plan_agent,
    build_subagents,
    run_plan,
)

# 构建 plan 模式的 deep agent：Planner + 4 个专用 subagent
agent = build_plan_agent(base_model())
print("Plan agent 构建完成：", type(agent).__name__)

## 示例 1：协作型任务（调研 + 写作 + 代码，三者可并行）

主管会把任务拆成三块，并在同一条消息里并行发起多个 `task` 调用。

In [ ]:
goal = (
    "帮我做三件事：\n"
    1. 调研 deepagents 的 subagent 机制（由 researcher 完成）；\n"
    2. 写一段 200 字的中文产品介绍（由 writer 完成）；\n"
    3. 给出一段最小可运行的 deepagents 创建示例（由 coder 完成）。\n"
    最后把三部分整合成一份简报。"
)

answer = run_plan(agent, goal, thread_id="plan-demo-1")
print(answer)

## 示例 2：带依赖的任务（先调研，再据结果撰写）

第二个子任务依赖第一个的结果，主管会等待 researcher 返回后再派发 writer。

In [ ]:
goal2 = (
    "先调研 RAG（检索增强生成）的核心流程与常见瓶颈，"
    再根据调研结论撰写一份面向技术经理的 150 字摘要，指出是否值得引入。"
)

answer2 = run_plan(agent, goal2, thread_id="plan-demo-2")
print(answer2)

## 自定义 subagent

你也可以只保留部分 subagent，或追加自己的专用 subagent，再传给 `build_plan_agent`。

In [ ]:
custom_subs = build_subagents(base_model())
# 例如：只保留 researcher 与 writer
custom_subs = [s for s in custom_subs if s["name"] in ("researcher", "writer")]

agent_custom = build_plan_agent(base_model(), subagents=custom_subs)
print("自定义 agent subagents:", [s["name"] for s in custom_subs])